# BatchNorm & Dropout

**Companion lesson:** https://ml-viz-ruby.vercel.app/courses/neural-networks/05-batchnorm-and-dropout

From-scratch implementations of BatchNorm and Dropout, with train/eval mode handling.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#1e293b'
plt.rcParams['figure.figsize'] = (8, 5)
np.random.seed(0)

## BatchNorm — worked example and forward pass

Normalize pre-activations to zero mean + unit variance per feature, then
rescale with learnable $\gamma, \beta$. Verify on the lesson's mini-batch $[2,4,6,8]$.

In [ ]:
import math

# Worked example from the lesson (pure stdlib, deterministic)
x_batch = [2.0, 4.0, 6.0, 8.0]
m = len(x_batch)
mu  = sum(x_batch) / m
var = sum((xi - mu)**2 for xi in x_batch) / m
std = math.sqrt(var)

x_hat = [(xi - mu) / std for xi in x_batch]
print('mu =', mu, '  var =', var, '  std =', round(std, 4))
print('x_hat:', [round(v, 4) for v in x_hat])
print('mean(x_hat):', round(sum(x_hat)/m, 12))
print('var(x_hat):', round(sum(v**2 for v in x_hat)/m, 12))

assert mu == 5.0 and var == 5.0
assert all(abs(sum(x_hat)/m) < 1e-12 for _ in [1])  # mean ~0
assert abs(sum(v**2 for v in x_hat)/m - 1.0) < 1e-12 # var ~1

gamma, beta = 2.0, 1.0
y_out = [gamma * xi + beta for xi in x_hat]
print()
print(f'After rescale (gamma={gamma}, beta={beta}):', [round(v, 4) for v in y_out])
print('VERIFY: BatchNorm worked example matches the lesson.')

## BatchNorm forward pass (numpy)

The production version handles a 2D mini-batch $(N, D)$, accumulates running statistics
for inference, and exposes train vs. eval mode.

In [ ]:
class BatchNorm:
    def __init__(self, num_features, eps=1e-5, momentum=0.1):
        self.gamma = np.ones(num_features)
        self.beta  = np.zeros(num_features)
        self.running_mean = np.zeros(num_features)
        self.running_var  = np.ones(num_features)
        self.eps = eps; self.momentum = momentum

    def forward(self, x, training=True):
        if training:
            mu  = x.mean(axis=0)
            var = x.var(axis=0)
            # update running statistics
            self.running_mean = (1-self.momentum)*self.running_mean + self.momentum*mu
            self.running_var  = (1-self.momentum)*self.running_var  + self.momentum*var
        else:
            mu, var = self.running_mean, self.running_var
        x_hat = (x - mu) / np.sqrt(var + self.eps)
        return self.gamma * x_hat + self.beta

D = 4
bn = BatchNorm(D)

# Training: 100 mini-batches of size 32 from a N(2, 3) distribution
for _ in range(100):
    x_train = np.random.randn(32, D) * np.sqrt(3) + 2
    out = bn.forward(x_train, training=True)

print('Running mean (should be ~2.0):', np.round(bn.running_mean, 3))
print('Running var  (should be ~3.0):', np.round(bn.running_var, 3))
print()

# Inference on a single example
x_inf = np.array([[3.0, 1.0, 5.0, 2.0]])
out_inf = bn.forward(x_inf, training=False)
print('Inference output (uses running stats):', np.round(out_inf, 4))

## Dropout — inverted dropout

Each neuron survives with probability $(1-p)$; surviving neurons are scaled by $1/(1-p)$
so the expected magnitude is unchanged. At inference, no scaling is needed.

In [ ]:
def dropout(h, p, training=True, seed=None):
    """Inverted dropout. h: activation array, p: drop probability."""
    if not training or p == 0.0:
        return h.copy()
    rng = np.random.RandomState(seed)
    mask = (rng.rand(*h.shape) > p).astype(float)
    return h * mask / (1.0 - p)

h = np.array([2.0, 1.5, 3.0, 0.5, 4.0, 2.5])
p = 0.5

# During training: ~50% of neurons zeroed; survivors scaled by 2
np.random.seed(42)
drops = [dropout(h, p, training=True) for _ in range(10000)]
means = np.stack(drops).mean(axis=0)

print('Original h:  ', h)
print('E[dropout(h)]:', np.round(means, 3), '(should ≈ h)')
assert np.allclose(means, h, atol=0.1), 'Expected value of dropout output should equal h'

# At inference: identity (no dropout applied)
out_inf = dropout(h, p, training=False)
assert np.allclose(out_inf, h), 'During inference dropout returns h unchanged'
print('Inference (no dropout):', out_inf)
print('VERIFY: E[dropout(h)] = h and inference is identity.')

## Effect on training: convergence and generalization

Simulate a simple regression task to show BatchNorm stabilizes convergence
and Dropout reduces overfitting.

In [ ]:
# Simulate activation distributions through a 5-layer network
# with and without BatchNorm, showing internal covariate shift
np.random.seed(1)
n_layers = 5
activation_means_plain = []
activation_means_bn    = []

x = np.random.randn(128, 32)

bn_layers = [BatchNorm(32) for _ in range(n_layers)]

for i in range(n_layers):
    W = np.random.randn(32, 32) * 2.0   # large init to cause shift
    x_plain = np.tanh(x @ W)
    x_bn    = np.tanh(bn_layers[i].forward(x @ W, training=True))
    activation_means_plain.append(abs(x_plain.mean()))
    activation_means_bn.append(abs(x_bn.mean()))
    x = x_plain  # carry plain forward

fig, ax = plt.subplots()
ax.plot(activation_means_plain, 'o-', color='#f59e0b', label='no BatchNorm')
ax.plot(activation_means_bn, 's--', color='#6366f1', label='with BatchNorm')
ax.set_xlabel('layer depth'); ax.set_ylabel('|mean activation|')
ax.set_title('Internal covariate shift: with vs. without BatchNorm')
ax.legend(); plt.show()

print('Without BN: mean activation drifts to', [round(v, 3) for v in activation_means_plain])
print('With BN:    mean activation stays ~0:', [round(v, 3) for v in activation_means_bn])

## The ensemble view of dropout

Each training forward pass samples a different 'thinned' network.
We visualise the diversity of predictions across 50 random masks.

In [ ]:
np.random.seed(3)
h_in = np.ones(20) * 2.0  # all neurons active at 2.0
p = 0.5

ensemble_outputs = []
for seed in range(500):
    h_drop = dropout(h_in, p, training=True, seed=seed)
    ensemble_outputs.append(h_drop.mean())

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(ensemble_outputs, bins=30, color='#6366f1', alpha=0.8)
ax.axvline(np.mean(ensemble_outputs), color='#f59e0b', lw=2, label=f'mean = {np.mean(ensemble_outputs):.3f}')
ax.axvline(h_in.mean(), color='#94a3b8', lw=2, ls='--', label=f'target = {h_in.mean():.1f}')
ax.set_xlabel('mean output'); ax.set_title('Dropout samples different "thinned networks"')
ax.legend(); plt.show()

print(f'Target: {h_in.mean():.1f}  |  Mean across 500 masks: {np.mean(ensemble_outputs):.4f}')
print('The mean of all thinned networks ≈ the full network output (inverted dropout property).')

## Key takeaways

- **BatchNorm** normalises each mini-batch to zero mean/unit variance, then re-scales with learned $\gamma, \beta$; use running stats at inference.
- **Dropout** randomly zeros $(p \times 100\%)$ of neurons per forward pass; survivors are scaled by $1/(1-p)$ (inverted dropout) so no scaling is needed at test time.
- BatchNorm addresses **optimisation instability**; Dropout addresses **overfitting**.
- Dropout ≈ training an **implicit ensemble** of $2^N$ thinned networks.

## ✏️ Your turn

### Exercise 1 — BatchNorm forward pass

Implement BatchNorm forward pass for a 2D mini-batch $(N, D)$. Verify mean ≈ 0 and variance ≈ 1 after normalisation, and that $\gamma=1, \beta=0$ leaves the output unchanged (but standardised).

In [ ]:
import numpy as np

def batchnorm_forward(x, gamma, beta, eps=1e-5):
    """BatchNorm forward pass.
    x: (N, D) mini-batch, gamma/beta: (D,) learnable parameters.
    Returns normalised output of shape (N, D)."""
    # TODO(you): compute per-feature mean and variance, normalise, then rescale
    ...

In [ ]:
np.random.seed(0)
N, D = 16, 4
x = np.random.randn(N, D) * 3 + 5   # mean≈5, std≈3
gamma = np.ones(D)
beta  = np.zeros(D)

out = batchnorm_forward(x, gamma, beta)

assert out.shape == x.shape, "output must have same shape as input"
assert np.allclose(out.mean(axis=0), 0.0, atol=1e-6), \
    "after BatchNorm, per-feature mean must be ≈ 0"
assert np.allclose(out.var(axis=0), 1.0, atol=1e-4), \
    "after BatchNorm, per-feature variance must be ≈ 1"

# With gamma=2, beta=1 the output should have mean≈1, std≈2
gamma2 = np.full(D, 2.0); beta2 = np.ones(D)
out2 = batchnorm_forward(x, gamma2, beta2)
assert np.allclose(out2.mean(axis=0), 1.0, atol=1e-6), \
    "with beta=1 the output mean should be 1"

# Edge case: a zero-variance (constant) feature column must stay finite
# (that's exactly what `eps` is for) and normalize to all zeros.
x_const = np.column_stack([x[:, 0], np.full(N, 7.0)])
gamma_c, beta_c = np.ones(2), np.zeros(2)
out_const = batchnorm_forward(x_const, gamma_c, beta_c)
assert np.all(np.isfinite(out_const)), "a constant column (var=0) must not divide by zero into NaN/Inf"
assert np.allclose(out_const[:, 1], 0.0, atol=1e-3), "a constant column normalizes to all zeros"

# Edge case: a batch of size 1 has zero variance by definition -- same guard
x_single = np.array([[3.0, -2.0, 5.0]])
out_single = batchnorm_forward(x_single, np.ones(3), np.zeros(3))
assert np.all(np.isfinite(out_single)), "a single-example batch must not blow up either"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def batchnorm_forward(x, gamma, beta, eps=1e-5):
    mu  = x.mean(axis=0)
    var = x.var(axis=0)
    x_hat = (x - mu) / np.sqrt(var + eps)
    return gamma * x_hat + beta
```

</details>

### Exercise 2 — Inverted dropout

Implement inverted dropout. Key invariant: the expected value of the output must equal the input — `E[dropout(h)] = h`.

In [ ]:
import numpy as np

def inverted_dropout(h, p, seed=None):
    """Inverted dropout: zero p-fraction of elements, scale survivors by 1/(1-p).
    h: 1-D or 2-D activation array, p: drop probability in [0,1).
    Returns array same shape as h."""
    # TODO(you): sample Bernoulli mask, multiply h, scale by 1/(1-p)
    ...

In [ ]:
h = np.array([1.0, 2.0, 3.0, 4.0, 5.0])

# Deterministic test: with seed, output is reproducible
out1 = inverted_dropout(h, p=0.5, seed=7)
out2 = inverted_dropout(h, p=0.5, seed=7)
assert np.allclose(out1, out2), "same seed must give same result"

# Zero inputs stay zero
zero_h = np.zeros(10)
assert np.all(inverted_dropout(zero_h, 0.5, seed=0) == 0), \
    "zero inputs must stay zero after dropout"

# Expected value test: mean over 10000 masks should ≈ h
samples = np.stack([inverted_dropout(h, 0.5, seed=i) for i in range(10000)])
assert np.allclose(samples.mean(axis=0), h, atol=0.15), \
    "E[dropout(h)] must equal h (inverted scaling) -- 0.15 tolerance, not 0.1, since the largest h value has the widest sampling spread at n=10000 draws"

# Edge case: p=0 -- nothing is dropped, output equals input exactly (rand() is
# never exactly 0.0 in practice, so the ">" survival test always keeps everyone)
assert np.array_equal(inverted_dropout(h, 0.0, seed=1), h), "p=0 drops nothing -- output equals input exactly"

# Edge case: p=1 -- every element is dropped, and 1/(1-p) divides by zero. This
# function doesn't guard against it (unlike the DropoutLayer class below), so
# it degrades to NaN rather than raising -- a good reason production code
# should validate p up front, which is exactly what DML #151 asks for next.
with np.errstate(divide='ignore', invalid='ignore'):
    degenerate = inverted_dropout(h, 1.0, seed=1)
assert np.all(np.isnan(degenerate)), "p=1 -- everything dropped, then 1/(1-p)=1/0 -> NaN"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def inverted_dropout(h, p, seed=None):
    rng = np.random.RandomState(seed)
    mask = (rng.rand(*h.shape) > p).astype(float)
    return h * mask / (1.0 - p)
```

</details>

### Exercise 3 — Running statistics for BatchNorm inference

BatchNorm accumulates running mean and variance during training (exponential moving average with momentum $\alpha$). At inference it uses these stored values instead of batch statistics. Implement the update and verify convergence.

In [ ]:
import numpy as np

def update_running_stats(running_mean, running_var, batch_mean, batch_var, momentum=0.1):
    """Update BatchNorm running statistics with one mini-batch.
    running_* and batch_*: scalar or array.
    Returns (new_running_mean, new_running_var)."""
    # TODO(you): exponential moving average: new = (1-momentum)*old + momentum*batch
    ...

In [ ]:
# After enough batches from N(3, 4), running stats should converge to mean=3, var=4
np.random.seed(42)
running_mean, running_var = 0.0, 1.0

for _ in range(500):
    batch = np.random.randn(32) * 2 + 3  # N(3, 4)
    running_mean, running_var = update_running_stats(
        running_mean, running_var, batch.mean(), batch.var())

assert abs(running_mean - 3.0) < 0.1, \
    f"running mean should converge to 3.0, got {running_mean:.4f}"
assert abs(running_var - 4.0) < 0.5, \
    f"running var should converge to 4.0, got {running_var:.4f}"

# Single update check: new = (1-0.1)*old + 0.1*batch
new_m, new_v = update_running_stats(0.0, 1.0, 5.0, 9.0, momentum=0.1)
assert abs(new_m - 0.5) < 1e-9, "0.9*0 + 0.1*5 = 0.5"
assert abs(new_v - 1.8) < 1e-9, "0.9*1 + 0.1*9 = 1.8"
print(f"Converged: running_mean = {running_mean:.3f}, running_var = {running_var:.3f}")

# Edge cases: momentum=0 (running stats never move) and momentum=1 (running
# stats jump straight to the batch stats, no smoothing at all)
rm0, rv0 = update_running_stats(1.0, 2.0, 99.0, 88.0, momentum=0.0)
assert rm0 == 1.0 and rv0 == 2.0, "momentum=0 -- running stats never move"

rm1, rv1 = update_running_stats(1.0, 2.0, 99.0, 88.0, momentum=1.0)
assert rm1 == 99.0 and rv1 == 88.0, "momentum=1 -- running stats jump straight to the batch stats"
print("✅ Exercise 3 passed")

<details>
<summary>💡 Show solution</summary>

```python
def update_running_stats(running_mean, running_var, batch_mean, batch_var, momentum=0.1):
    new_mean = (1 - momentum) * running_mean + momentum * batch_mean
    new_var  = (1 - momentum) * running_var  + momentum * batch_var
    return new_mean, new_var
```

</details>

---
## 🔬 Extra practice — a stateful dropout layer (DML #151)

The `inverted_dropout()` function above is stateless -- great for demonstrating
the math, but it can't be reused across a `forward` then `backward` call in a
training loop without the caller manually threading the mask through. [DML
#151](https://github.com/Open-Deep-ML/DML-OpenProblem/tree/main/questions/151_dropout-layer)
asks for a small `DropoutLayer` class instead: it validates its dropout rate
once at construction time, remembers the mask it drew on the last *training*
forward pass, and reuses that exact mask in `backward()` -- which is exactly
the p=1 failure mode you just triggered above, guarded against up front.

### Exercise 5 — a `DropoutLayer` class (DML #151)

Implement `__init__` (reject an invalid `p`), `forward` (pass through
unchanged at inference or when `p == 0`; otherwise draw a fresh Bernoulli-keep
mask, scale it by `1/(1-p)`, store it, and apply it), and `backward` (re-apply
the stored mask to the incoming gradient).

In [ ]:
class DropoutLayer:
    """DML #151: inverted dropout as a layer with its own forward/backward pass."""

    def __init__(self, p: float):
        # TODO(you): reject an invalid p (raise ValueError if not 0 <= p < 1 --
        # p == 1 would divide by zero, exactly the failure you triggered above),
        # then store p and initialize self.mask = None
        ...

    def forward(self, x: np.ndarray, training: bool = True) -> np.ndarray:
        x = np.asarray(x, dtype=float)
        if not training or self.p == 0.0:
            return x.copy()
        # TODO(you): draw a Bernoulli-keep mask -- np.random.binomial(1, 1 - self.p, size=x.shape) --
        # scale it by 1/(1-p), store it on self.mask, and apply it to x
        return ...

    def backward(self, grad: np.ndarray) -> np.ndarray:
        # TODO(you): re-apply the mask saved by the last training forward() call
        return ...

In [ ]:
# Checks — run me (DML's own published test cases, from tests.json)
np.random.seed(42)
x = np.array([[1.0, 2.0], [3.0, 4.0]])
grad = np.array([[0.5, 0.2], [1.0, 2.0]])

layer = DropoutLayer(0.2)
out_train = layer.forward(x, training=True)
out_eval = layer.forward(x, training=False)
out_back = layer.backward(grad)

assert np.allclose(out_train, [[1.25, 0.0], [3.75, 5.0]]), "DML's own worked example (seed 42, p=0.2)"
assert np.allclose(out_eval, x), "inference mode passes input through unchanged"
assert np.allclose(out_back, [[0.625, 0.0], [1.25, 2.5]]), \
    "backward reuses the SAME mask as the last training forward pass -- not the eval one in between"

# The mask changes on every training call (a fresh Bernoulli draw each time)
np.random.seed(42)
big = np.ones((1000, 1000))
layer2 = DropoutLayer(0.2)
_ = layer2.forward(big, training=True); mask1 = layer2.mask.copy()
_ = layer2.forward(big, training=True); mask2 = layer2.mask.copy()
assert not np.array_equal(mask1, mask2), "each training forward pass draws a fresh mask"

# The expected-value property holds for other p too
np.random.seed(42)
out_p3 = DropoutLayer(0.3).forward(np.ones((1000, 1000)), training=True)
assert abs(out_p3.mean() - 1.0) < 0.1, "E[dropout(h)] ≈ h -- inverted scaling keeps the mean unbiased"

# Edge cases: p=0 (never drops, even in training) and p=1 / out-of-range p
# (rejected outright at construction, not left to silently produce NaN)
layer_p0 = DropoutLayer(0.0)
assert np.array_equal(layer_p0.forward(x, training=True), x), "p=0 is a no-op even in training mode"

for bad_p in [1.0, 1.5, -0.5]:
    try:
        DropoutLayer(bad_p)
        raise AssertionError(f"p={bad_p} should have raised ValueError")
    except ValueError:
        pass
print("✅ DML 151 dropout-layer passed")

<details>
<summary>💡 Show solution</summary>

```python
class DropoutLayer:
    def __init__(self, p: float):
        if not (0 <= p < 1):
            raise ValueError("p must be in [0, 1)")
        self.p = p
        self.mask = None

    def forward(self, x: np.ndarray, training: bool = True) -> np.ndarray:
        x = np.asarray(x, dtype=float)
        if not training or self.p == 0.0:
            return x.copy()
        keep = np.random.binomial(1, 1 - self.p, size=x.shape).astype(float)
        self.mask = keep / (1 - self.p)
        return x * self.mask

    def backward(self, grad: np.ndarray) -> np.ndarray:
        return grad * self.mask
```

</details>

### Beyond this lesson: normalization-free transformers (DML #128)

BatchNorm and Dropout both assume batch statistics are worth computing -- but a
2024 line of work asks whether you need normalization at all. [DyT (Dynamic
Tanh)](https://github.com/Open-Deep-ML/DML-OpenProblem/tree/main/questions/128_dynamic-tanh-normalization-free-transformer-activa)
replaces LayerNorm in Transformer blocks with a single learned scalar $\alpha$
inside a squashing nonlinearity: $\text{DyT}(x) = \gamma \odot \tanh(\alpha x) + \beta$ --
no mean/variance computed over a batch or layer at all. It's a nice reminder
that "keep activations well-scaled" and "compute batch/layer statistics to do
it" aren't the same requirement (see `transformers/03-transformer-architecture.ipynb`
for where LayerNorm itself is covered).